# Build cell_features (Week 3)

Hand-crafted early-cycle features from `cycle_summary.csv`, merged with `voltage_features.csv`.

Labels (`EOL`, `initial_capacity`) come from `cell_targets.csv`.

Output: `data/processed/cell_features.csv` (134 rows, one per kept cell).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
TARGETS_PATH = ROOT / 'data' / 'cell_targets.csv'
SUMMARY_PATH = ROOT / 'data' / 'processed' / 'cycle_summary.csv'
VOLTAGE_PATH = ROOT / 'data' / 'processed' / 'voltage_features.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'cell_features.csv'

WINDOWS = (20, 50, 100)
CAPACITY_CYCLES = (10, 50, 100)

print('Project root:', ROOT)
print('Output:', OUTPUT_PATH)

Project root: /Users/levonl/Dev/gnem-battery-degradation
Output: /Users/levonl/Dev/gnem-battery-degradation/data/processed/cell_features.csv


In [2]:
def value_at_cycle(group: pd.DataFrame, cycle: int, column: str) -> float:
    row = group.loc[group['cycle_index'] == cycle, column]
    return float(row.iloc[0]) if len(row) else np.nan


def slope(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2 or np.allclose(x, x[0]):
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def extract_summary_features(group: pd.DataFrame, initial_capacity: float) -> dict:
    g = group[group['cycle_index'] >= 1].sort_values('cycle_index')
    features = {}

    features['resistance_initial'] = value_at_cycle(g, 1, 'dc_internal_resistance')

    for cycle in CAPACITY_CYCLES:
        cap = value_at_cycle(g, cycle, 'discharge_capacity')
        features[f'capacity_c{cycle}'] = cap
        features[f'soh_c{cycle}'] = cap / initial_capacity if np.isfinite(cap) else np.nan

    for window in WINDOWS:
        w = g[g['cycle_index'] <= window]
        prefix = f'w{window}'
        if w.empty:
            for key in (
                f'capacity_slope_{prefix}',
                f'capacity_mean_{prefix}',
                f'capacity_std_{prefix}',
                f'soh_{prefix}',
                f'resistance_mean_{prefix}',
                f'resistance_slope_{prefix}',
                f'efficiency_mean_{prefix}',
                f'efficiency_std_{prefix}',
                f'temp_mean_{prefix}',
            ):
                features[key] = np.nan
            continue

        cyc = w['cycle_index'].to_numpy(dtype=float)
        cap = w['discharge_capacity'].to_numpy(dtype=float)
        features[f'capacity_slope_{prefix}'] = slope(cyc, cap)
        features[f'capacity_mean_{prefix}'] = float(np.nanmean(cap))
        features[f'capacity_std_{prefix}'] = float(np.nanstd(cap))

        cap_at_w = value_at_cycle(w, min(window, int(w['cycle_index'].max())), 'discharge_capacity')
        features[f'soh_{prefix}'] = cap_at_w / initial_capacity if np.isfinite(cap_at_w) else np.nan

        resistance = w['dc_internal_resistance'].to_numpy(dtype=float)
        ok_r = np.isfinite(resistance)
        features[f'resistance_mean_{prefix}'] = float(np.nanmean(resistance)) if ok_r.any() else np.nan
        features[f'resistance_slope_{prefix}'] = (
            slope(cyc[ok_r], resistance[ok_r]) if ok_r.sum() >= 2 else np.nan
        )

        efficiency = w['energy_efficiency'].to_numpy(dtype=float)
        ok_e = np.isfinite(efficiency)
        features[f'efficiency_mean_{prefix}'] = float(np.nanmean(efficiency)) if ok_e.any() else np.nan
        features[f'efficiency_std_{prefix}'] = float(np.nanstd(efficiency)) if ok_e.sum() >= 2 else np.nan

        temp = w['temperature_average'].to_numpy(dtype=float)
        ok_t = np.isfinite(temp)
        features[f'temp_mean_{prefix}'] = float(np.nanmean(temp)) if ok_t.any() else np.nan

    return features

In [3]:
targets = pd.read_csv(TARGETS_PATH)
summary = pd.read_csv(SUMMARY_PATH)
voltage = pd.read_csv(VOLTAGE_PATH)

initial_caps = targets.set_index('file_id')['initial_capacity']
summary_parts = []
for file_id, group in summary.groupby('file_id', sort=True):
    summary_parts.append(
        {
            'file_id': file_id,
            'cell_id': group['cell_id'].iloc[0],
            **extract_summary_features(group, initial_caps[file_id]),
        }
    )

summary_features = pd.DataFrame(summary_parts)
cell_features = (
    targets.merge(summary_features, on=['file_id', 'cell_id'], how='inner')
    .merge(voltage, on=['file_id', 'cell_id'], how='inner')
)

label_cols = ['file_id', 'cell_id', 'EOL', 'initial_capacity']
feature_cols = [c for c in cell_features.columns if c not in label_cols]
cell_features = cell_features[label_cols + feature_cols]

print(f"Rows: {len(cell_features)}")
print(f"Summary features: {len(summary_features.columns) - 2}")
print(f"Voltage features: {len(voltage.columns) - 2}")
print(f"Total feature columns: {len(feature_cols)}")

Rows: 134
Summary features: 34
Voltage features: 10
Total feature columns: 44


In [4]:
assert len(cell_features) == len(targets) == len(voltage)
assert cell_features['file_id'].is_unique
assert cell_features[feature_cols].isna().sum().sum() == 0

print('Missing values (should be 0 for current dataset):')
print(cell_features[feature_cols].isna().sum().sort_values(ascending=False).head())
cell_features.head()

Missing values (should be 0 for current dataset):
resistance_initial     0
capacity_c10           0
temp_mean_w50          0
capacity_slope_w100    0
capacity_mean_w100     0
dtype: int64


,file_id,cell_id,EOL,initial_capacity,resistance_initial,capacity_c10,soh_c10,capacity_c50,soh_c50,capacity_c100,...,delta_v_mean_c10_c50,delta_v_std_c10_c50,delta_v_var_c10_c50,delta_v_min_c10_c50,delta_v_max_c10_c50,delta_v_mean_c10_c100,delta_v_std_c10_c100,delta_v_var_c10_c100,delta_v_min_c10_c100,delta_v_max_c10_c100
0,FastCharge_000015_CH4_structure.json,el150800737390,796,1.058157,0.015554,1.063230,1.004794,1.063544,1.005091,1.061757,...,-0.001335,0.003679,0.000014,-0.020201,0.016586,-0.006345,0.009476,0.000090,-0.056352,0.007565
1,FastCharge_000002_CH18_structure.json,el150800737233,828,1.063661,0.015485,1.067935,1.004019,1.068569,1.004614,1.067026,...,-0.001761,0.004507,0.000020,-0.025450,0.019347,-0.008084,0.011135,0.000124,-0.065183,0.013658
2,FastCharge_000002_CH10_structure.json,el150800737329,1009,1.066573,0.015435,1.070419,1.003606,1.070960,1.004113,1.069454,...,-0.000816,0.004040,0.000016,-0.023264,0.021749,-0.006161,0.009935,0.000099,-0.059128,0.031566
3,FastCharge_000039_CH27_structure.json,EL150800460507,860,1.075039,0.016832,1.080971,1.005518,1.080980,1.005526,1.079695,...,-0.005427,0.006914,0.000048,-0.040312,0.014960,-0.011326,0.013732,0.000189,-0.079695,0.022920
4,FastCharge_000013_CH14_structure.json,EL150800463882,788,1.050799,0.016517,1.058100,1.006948,1.060798,1.009515,1.059972,...,-0.002445,0.003068,0.000009,-0.014730,0.022695,-0.007840,0.009085,0.000083,-0.050664,-0.000045


In [5]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
cell_features.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH}: {len(cell_features)} rows, {len(feature_cols)} feature columns")

Wrote /Users/levonl/Dev/gnem-battery-degradation/data/processed/cell_features.csv: 134 rows, 44 feature columns
